# 03 — Lazy Pipelines

Generators get really useful when you **chain them**. Each stage takes an iterable in, yields one transformed item out. Memory stays flat, you can stop anywhere, and each stage is small enough to test in isolation.

This is the shape under tools like `itertools`, log processors, ETL pipelines, and (later) streaming LLM responses.

## The classic shape — read → clean → parse → filter → sum

In [1]:
SAMPLE = """\
# a comment
10
20

not-a-number
30
# another
-5
40
"""

def read_lines(text):
    yield from text.splitlines()

def strip_blanks_and_comments(lines):
    for line in lines:
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        yield line

def safe_parse_int(lines):
    for line in lines:
        try:
            yield int(line)
        except ValueError:
            # skip junk silently — a stricter pipeline could raise / record instead
            continue

def positives(ints):
    for n in ints:
        if n > 0:
            yield n

# Compose the pipeline. Nothing has run yet — each call returns a generator.
lines    = read_lines(SAMPLE)
cleaned  = strip_blanks_and_comments(lines)
parsed   = safe_parse_int(cleaned)
pos      = positives(parsed)

# The pipeline runs when SOMETHING consumes it. `sum` is the driver here.
print("total:", sum(pos))

total: 100


**Trace it in your head:** `sum` asks `pos` for the next int → `pos` asks `parsed` → `parsed` asks `cleaned` → `cleaned` asks `lines` → `lines` yields one line. The data flows *one item at a time*; no intermediate list is built.

## Why bother — memory and early exit

Pretend the input is a 10 GB log file. The pipeline above streams through it; `sum` never sees more than one int at a time. And if you only wanted the *first* matching item, the pipeline stops the moment you ask for it.

In [2]:
# Just the first positive — pipeline stops immediately after it's found.
from itertools import islice

first_positive = next(positives(safe_parse_int(strip_blanks_and_comments(read_lines(SAMPLE)))))
print("first positive:", first_positive)

# Or the first 3:
print("first 3 positives:",
      list(islice(positives(safe_parse_int(strip_blanks_and_comments(read_lines(SAMPLE)))), 3)))

first positive: 10
first 3 positives: [10, 20, 30]


## `itertools` — the standard toolkit

Most pipeline plumbing already exists in `itertools`. Read its docs once — you'll keep coming back.

In [3]:
from itertools import islice, takewhile, dropwhile, chain, count, cycle, groupby

# islice — slice an iterator without converting to a list first
print(list(islice(count(1), 5)))            # [1, 2, 3, 4, 5]
print(list(islice(count(1), 5, 10)))        # [6, 7, 8, 9, 10]

# takewhile / dropwhile — early-stop / skip-prefix conditions
print(list(takewhile(lambda x: x < 5, [1, 2, 3, 7, 1, 0])))   # [1, 2, 3]  — stops at 7
print(list(dropwhile(lambda x: x < 5, [1, 2, 3, 7, 1, 0])))   # [7, 1, 0]

# chain — concatenate without copying
print(list(chain([1, 2], (3, 4), "ab")))

# groupby — adjacent-equal grouping (sort first if you want global groups)
data = [("a", 1), ("a", 2), ("b", 3), ("b", 4), ("a", 5)]
for key, group in groupby(data, key=lambda x: x[0]):
    print(key, list(group))

[1, 2, 3, 4, 5]
[6, 7, 8, 9, 10]
[1, 2, 3]
[7, 1, 0]
[1, 2, 3, 4, 'a', 'b']
a [('a', 1), ('a', 2)]
b [('b', 3), ('b', 4)]
a [('a', 5)]


## Lazy file reading — Python's most useful generator

`open(...)` returns a file object that's itself an iterator over lines. It's already lazy — perfect input for a pipeline. Demo below uses a temp file.

In [4]:
import tempfile, os

# Write a small file so we have something to read.
tmp = tempfile.NamedTemporaryFile("w", delete=False, encoding="utf-8")
tmp.write("INFO  starting\n")
tmp.write("DEBUG noise 1\n")
tmp.write("ERROR oops 1\n")
tmp.write("DEBUG noise 2\n")
tmp.write("ERROR oops 2\n")
tmp.write("INFO  done\n")
tmp.close()

def only(prefix, lines):
    for line in lines:
        if line.startswith(prefix):
            yield line

with open(tmp.name) as f:
    # `f` is the iterator. The whole pipeline never holds more than one line at a time.
    errors = only("ERROR", f)
    for err in errors:
        print(err.rstrip())

os.unlink(tmp.name)

ERROR oops 1
ERROR oops 2


## Don't accidentally collapse the laziness

One `list(...)` in the middle of the pipeline materializes everything up to that point. Useful sometimes (e.g. before passing to a function that needs to peek twice), but often a mistake.

In [5]:
# Lazy: each line is processed and discarded.
def lazy_count(text):
    return sum(1 for line in text.splitlines() if line.startswith("ERROR"))

# Eager: builds an intermediate list of every line first.
def eager_count(text):
    lines = list(text.splitlines())          # full list in memory
    return sum(1 for line in lines if line.startswith("ERROR"))

blob = "\n".join(["INFO x"] * 1000 + ["ERROR y"] * 5)
print(lazy_count(blob), eager_count(blob))   # same result, very different memory profile

5 5


## Where this pattern shows up later

- **`async for`** in folder 11: same shape with `async def` + `yield`.
- **Streaming LLM responses**: each token is yielded as it arrives — the agent code consumes it through exactly this pattern.
- **DB cursors / API pagination**: a generator that fetches more pages lazily, transparent to the consumer.
- **FastAPI dependencies with `yield`** (folder 14): a `setup → yield → teardown` shape backed by a generator.

## Mini-exercise

1. Extend the `read → clean → parse → positives` pipeline with a `windowed(n)` stage that yields rolling windows of the last `n` numbers (use `collections.deque(maxlen=n)`). Print rolling averages.
2. Replace the manual `take_first_k` you wrote with `itertools.islice`. Confirm the rest of the pipeline still works.
3. Predict, then run:
   ```python
   pipeline = (x*x for x in range(10))
   filtered = (x for x in pipeline if x > 20)
   print(sum(filtered))
   print(sum(filtered))   # ?
   ```
   Why does the second `sum` return 0?

In [6]:
# your solutions here


**Takeaways**

- Compose generators: each stage is `iterable in → iterable out`. Memory stays flat.
- Nothing runs until a consumer pulls (`sum`, `for`, `next`, `list`, `any` ...).
- `itertools` covers most plumbing — `islice`, `takewhile`, `chain`, `groupby`, ...
- An accidental `list(...)` mid-pipeline kills the laziness. Be deliberate.

Next: [04_event_pipeline/](04_event_pipeline/) — a real .py module wiring this up.